# Drone Object Detection — Experiment Results Analysis

This notebook loads results from all trained models and produces comparison plots and tables.

**Setup:** Place each model folder from Drive into this directory (`colab/results/`) so the
structure looks like:

```
colab/results/
    yolov8n/
        results.csv
        run_info.json
        weights/
        confusion_matrix_normalized.png
        PR_curve.png
        ...
    yolov8s/   (same structure)
    yolov8m/   (same structure)
    yolov8n_tuned/  (same structure)
```

Then run all cells top to bottom.

In [ ]:
# -- Cell 1: Imports and Load All Model Results ------------------------------
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import display

RESULTS_DIR = Path('.')

model_data = {}
for model_dir in sorted(RESULTS_DIR.iterdir()):
    if not model_dir.is_dir():
        continue
    info_path = model_dir / 'run_info.json'
    csv_path  = model_dir / 'results.csv'
    if not info_path.exists() or not csv_path.exists():
        continue
    with open(info_path) as fh:
        info = json.load(fh)
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    model_data[model_dir.name] = {'info': info, 'df': df}

if not model_data:
    print('ERROR: No model results found.')
    print('Place model folders (yolov8n/, yolov8s/, etc.) into this directory and re-run.')
else:
    print(f'Loaded {len(model_data)} model(s): {list(model_data.keys())}')

## Results Summary

The table below shows the best-epoch metrics for each model, along with training cost information.
**Bold** values indicate the best result in each column.

In [ ]:
# -- Cell 2: Summary Table ---------------------------------------------------
rows = []
for name, data in model_data.items():
    info = data['info']
    bm   = info['best_metrics']
    t_sec = info.get('training_time_seconds') or 0
    rows.append({
        'Model':             name,
        'mAP50':             bm['mAP50'],
        'mAP50-95':          bm['mAP50_95'],
        'Precision':         bm['precision'],
        'Recall':            bm['recall'],
        'Best Epoch':        info.get('best_epoch'),
        'Total Epochs':      info.get('epochs_completed'),
        'Train Time (min)':  round(t_sec / 60, 1),
        'Size (MB)':         info.get('model_size_mb'),
    })

summary_df = pd.DataFrame(rows).set_index('Model')

def bold_max(s):
    is_max = s == s.max()
    return ['font-weight: bold' if v else '' for v in is_max]

display(
    summary_df.style
        .apply(bold_max, subset=['mAP50', 'mAP50-95', 'Precision', 'Recall'])
        .format({
            'mAP50': '{:.4f}',
            'mAP50-95': '{:.4f}',
            'Precision': '{:.4f}',
            'Recall': '{:.4f}',
        })
)

## mAP Comparison

Grouped bar chart comparing mAP50 and mAP50-95 at each model's best epoch.
mAP50 is the primary metric; mAP50-95 provides a stricter secondary view.

In [ ]:
# -- Cell 3: mAP Comparison Bar Chart ----------------------------------------
models  = list(model_data.keys())
map50   = [model_data[m]['info']['best_metrics']['mAP50']    for m in models]
map5095 = [model_data[m]['info']['best_metrics']['mAP50_95'] for m in models]

x     = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, map50,   width, label='mAP50',    color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, map5095, width, label='mAP50-95', color='darkorange', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Model')
ax.set_ylabel('mAP')
ax.set_title('Model Comparison -- mAP50 and mAP50-95 (best epoch)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_map.png')

## Training Curves and Overfitting

Each subplot shows training loss vs validation loss over all epochs for one model.
A large and widening gap between the two curves indicates **overfitting**.
If both curves are high and flat, the model is **underfitting** (too little capacity or too few epochs).
The green dashed line marks the epoch with the best validation mAP50.

In [ ]:
# -- Cell 4: Training vs Validation Loss Curves (2x2 grid) ------------------
models  = list(model_data.keys())
n_cols  = 2
n_rows  = (len(models) + 1) // 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
axes = np.array(axes).flatten()

for i, (name, data) in enumerate(model_data.items()):
    ax   = axes[i]
    df   = data['df']
    info = data['info']
    best_ep = info.get('best_epoch', 1)
    epochs  = df.index + 1

    train_col = 'train/box_loss'
    val_col   = 'val/box_loss'

    if train_col in df.columns and val_col in df.columns:
        ax.plot(epochs, df[train_col], label='Train box loss', color='royalblue',  linewidth=1.5)
        ax.plot(epochs, df[val_col],   label='Val box loss',   color='tomato',     linewidth=1.5)
        ax.axvline(x=best_ep, color='green', linestyle='--', alpha=0.7,
                   label=f'Best epoch ({best_ep})')
    else:
        ax.text(0.5, 0.5, 'Loss columns not found\nCheck results.csv headers',
                transform=ax.transAxes, ha='center', va='center')

    map50 = info['best_metrics']['mAP50']
    ax.set_title(f'{name}  (best mAP50 = {map50:.3f})')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Box Loss')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Training vs Validation Box Loss -- Overfitting / Underfitting View', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## Convergence Comparison

mAP50 over epochs for all models on one plot. Shows how quickly each model learns
and whether it has plateaued or is still improving at the final epoch.

In [ ]:
# -- Cell 5: mAP50 Convergence (All Models) ----------------------------------
fig, ax = plt.subplots(figsize=(11, 5))

colors = ['royalblue', 'darkorange', 'green', 'crimson', 'purple', 'brown']

for i, (name, data) in enumerate(model_data.items()):
    df = data['df']
    col = 'metrics/mAP50(B)'
    if col in df.columns:
        ax.plot(df.index + 1, df[col],
                label=name, color=colors[i % len(colors)], linewidth=1.8)

ax.set_xlabel('Epoch')
ax.set_ylabel('mAP50 (validation)')
ax.set_title('mAP50 Convergence -- All Models')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0, None)

plt.tight_layout()
plt.savefig('convergence_map50.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: convergence_map50.png')

## Overfitting Analysis

The generalization gap is computed as `val_box_loss - train_box_loss` at each model's best epoch.
A larger gap means the model memorized training data without generalizing.
A gap near zero (or negative) means the model generalizes well to unseen examples.

In [ ]:
# -- Cell 6: Overfitting Gap Bar Chart ----------------------------------------
models = list(model_data.keys())
gaps   = []

for name in models:
    df   = model_data[name]['df']
    info = model_data[name]['info']
    idx  = info.get('best_epoch', 1) - 1

    if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
        train_loss = float(df.loc[idx, 'train/box_loss'])
        val_loss   = float(df.loc[idx, 'val/box_loss'])
        gaps.append(val_loss - train_loss)
    else:
        gaps.append(float('nan'))

bar_colors = ['green' if g < 0.05 else 'darkorange' if g < 0.15 else 'crimson'
              for g in gaps]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(models, gaps, color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

for bar, gap in zip(bars, gaps):
    if not np.isnan(gap):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                f'{gap:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Model')
ax.set_ylabel('Val loss - Train loss (at best epoch)')
ax.set_title('Overfitting Indicator -- Generalization Gap\n(smaller gap = better generalization)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('overfitting_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: overfitting_gap.png')
print()
print('Green  (< 0.05): good generalization')
print('Orange (0.05-0.15): mild overfitting')
print('Red    (> 0.15): significant overfitting')

## Model Efficiency

Accuracy (mAP50) vs model size in MB. Point size is proportional to training time.
The ideal model is in the top-left: high accuracy, small size, fast to train.

In [ ]:
# -- Cell 7: Efficiency Scatter Plot -----------------------------------------
models    = list(model_data.keys())
sizes     = [model_data[m]['info'].get('model_size_mb') or 0   for m in models]
map50s    = [model_data[m]['info']['best_metrics']['mAP50']     for m in models]
times_min = [(model_data[m]['info'].get('training_time_seconds') or 0) / 60
             for m in models]

point_sizes = [max(60, t * 4) for t in times_min]

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(sizes, map50s, s=point_sizes, alpha=0.75,
                     c=range(len(models)), cmap='tab10',
                     edgecolors='black', linewidths=0.5)

for i, name in enumerate(models):
    ax.annotate(name, (sizes[i], map50s[i]),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel('Model Size (MB)')
ax.set_ylabel('mAP50 (best epoch)')
ax.set_title('Accuracy vs. Model Size\n(point size ~ training time in minutes)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('efficiency_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: efficiency_scatter.png')

## Cost Analysis and Final Recommendation

Training cost is estimated using the approximate Google Colab GPU compute unit rate.
The best model is selected by mAP50; the most efficient model maximizes mAP50 per MB of size.

In [ ]:
# -- Cell 8: Cost Analysis Table and Recommendation -------------------------
COLAB_GPU_COST_PER_HR = 0.50  # approximate Colab compute unit cost in USD

rows = []
for name, data in model_data.items():
    info    = data['info']
    t_sec   = info.get('training_time_seconds') or 0
    t_min   = round(t_sec / 60, 1)
    cost    = round((t_sec / 3600) * COLAB_GPU_COST_PER_HR, 3)
    size_mb = info.get('model_size_mb')
    map50   = info['best_metrics']['mAP50']
    eff     = round(map50 / size_mb, 4) if size_mb else None
    rows.append({
        'Model':            name,
        'Train Time (min)': t_min,
        'Model Size (MB)':  size_mb,
        'Est. Cost ($)':    cost,
        'mAP50':            map50,
        'mAP50 / MB':       eff,
    })

cost_df = pd.DataFrame(rows).set_index('Model')
display(cost_df)

# Best by accuracy
best_map = max(model_data, key=lambda m: model_data[m]['info']['best_metrics']['mAP50'])

# Best by efficiency (mAP50 per MB)
candidates = [m for m in model_data if model_data[m]['info'].get('model_size_mb')]
best_eff   = max(candidates,
                 key=lambda m: (model_data[m]['info']['best_metrics']['mAP50'] /
                                model_data[m]['info']['model_size_mb']))

bm_best  = model_data[best_map]['info']['best_metrics']
eff_val  = round(model_data[best_eff]['info']['best_metrics']['mAP50'] /
                 model_data[best_eff]['info']['model_size_mb'], 4)

print()
print(f'Best model by mAP50:       {best_map}  (mAP50={bm_best["mAP50"]})')
print(f'Best model by mAP50/MB:    {best_eff}  (efficiency={eff_val})')

## Auto-Generated YOLOv8 Plots

YOLOv8 automatically saves confusion matrices, precision-recall curves, and training
summary charts. The cell below displays those plots for the best-performing model.

In [ ]:
# -- Cell 9: Display Auto-Generated Plots for Best Model --------------------
best_map  = max(model_data, key=lambda m: model_data[m]['info']['best_metrics']['mAP50'])
plots_dir = RESULTS_DIR / best_map

plot_files = [
    ('results.png',                     'Training Summary (all metrics over epochs)'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix (normalized)'),
    ('PR_curve.png',                    'Precision-Recall Curve'),
    ('F1_curve.png',                    'F1 Confidence Curve'),
]

found = [(fn, title) for fn, title in plot_files if (plots_dir / fn).exists()]

if not found:
    print(f'No auto-generated plots found in {plots_dir}/')
    print('Expected files: results.png, confusion_matrix_normalized.png, PR_curve.png, F1_curve.png')
else:
    print(f'Best model: {best_map} -- displaying {len(found)} auto-generated plot(s)')
    for fname, title in found:
        img = mpimg.imread(str(plots_dir / fname))
        fig, ax = plt.subplots(figsize=(11, 6))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'{best_map} -- {title}', fontsize=12)
        plt.tight_layout()
        plt.show()

## Summary and Conclusion

### Metric Selection
**mAP50** was selected as the primary metric because:
1. It balances precision and recall simultaneously across all confidence thresholds.
2. It weights the three classes (person, vehicle, two-wheeler) equally, preventing
   the majority class (vehicle, 54%) from dominating evaluation.
3. It is the standard benchmark metric for VisDrone and COCO-style detection tasks,
   enabling comparison with published results.

### Cross-Validation
A scene-level three-way **train / validation / test** split was used, consistent with
COCO benchmark standards. The test set was held out throughout all model selection and
hyperparameter tuning, ensuring an unbiased final evaluation.

### Overfitting and Underfitting
The training/validation loss curves and generalization gap charts above show the
generalization behavior of each model. Models with a small, stable gap between
training and validation loss at the best epoch generalize well from training data
to unseen examples.

### Model Recommendation
Based on the metrics above:
- The model with the **highest mAP50** is the best performer on this task.
- The model with the **highest mAP50/MB** is the most efficient for deployment.
- The training curves confirm whether the selected model is well-generalized
  (small gap) or requires further regularization.